In [1]:
import pandas as pd
import numpy as np

# ---------------------------
# Load the per-user feature table from Step 1
# ---------------------------
user_features = pd.read_csv("phase5_user_rfm_features.csv")  # adjust filename if yours differs

print("Loaded shape:", user_features.shape)

# ---------------------------
# Drop dead-weight / redundant columns
# total_removes: zero variance across the dataset, adds nothing
# net_cart_activity ≈ total_carts since total_removes is ~0, so keep only one
# purchase_sessions: redundant with purchase_rate (rate = purchase_sessions / total_sessions)
#   -> keep purchase_sessions for DISPLAY/profiling, but exclude from clustering inputs later
# avg_hour_of_day: cyclical, not a value/behavior signal -> exclude from clustering inputs,
#   keep for profiling ("this segment shops late evening")
# ---------------------------
user_features = user_features.drop(columns=['total_removes'], errors='ignore')

# ---------------------------
# Split into buyers vs non-buyers
# ---------------------------
buyers = user_features[user_features['purchase_sessions'] > 0].copy()
non_buyers = user_features[user_features['purchase_sessions'] == 0].copy()

print(f"\nBuyers: {len(buyers):,} ({len(buyers)/len(user_features)*100:.2f}%)")
print(f"Non-buyers: {len(non_buyers):,} ({len(non_buyers)/len(user_features)*100:.2f}%)")

# ---------------------------
# Log-transform heavy-tailed columns (log1p handles zeros safely)
# Applied separately to each group since their distributions differ
# ---------------------------
skewed_cols_common = [
    'total_sessions', 'total_events', 'total_views',
    'avg_session_duration', 'total_carts', 'net_cart_activity'
]
skewed_cols_buyers_only = ['estimated_total_value', 'estimated_avg_value']

for col in skewed_cols_common:
    buyers[f'log_{col}'] = np.log1p(buyers[col])
    non_buyers[f'log_{col}'] = np.log1p(non_buyers[col])

for col in skewed_cols_buyers_only:
    buyers[f'log_{col}'] = np.log1p(buyers[col])
    # non_buyers will have these as 0 anyway (no purchases), skip log for them

print("\nBuyers columns now:", list(buyers.columns))
print("\nLog-transform sanity check (buyers, log_total_sessions):")
print(buyers['log_total_sessions'].describe())

# Save intermediate outputs so Cell 2/3 can be re-run independently if needed
buyers.to_csv("phase5_buyers_features.csv", index=False)
non_buyers.to_csv("phase5_non_buyers_features.csv", index=False)
print("\nSaved phase5_buyers_features.csv and phase5_non_buyers_features.csv")

Loaded shape: (5316561, 17)

Buyers: 690,803 (12.99%)
Non-buyers: 4,625,758 (87.01%)

Buyers columns now: ['user_id', 'total_sessions', 'purchase_sessions', 'total_events', 'total_views', 'avg_session_duration', 'total_carts', 'avg_unique_products', 'avg_unique_categories', 'avg_brands', 'avg_hour_of_day', 'estimated_total_value', 'estimated_avg_value', 'recency_days', 'purchase_rate', 'net_cart_activity', 'log_total_sessions', 'log_total_events', 'log_total_views', 'log_avg_session_duration', 'log_total_carts', 'log_net_cart_activity', 'log_estimated_total_value', 'log_estimated_avg_value']

Log-transform sanity check (buyers, log_total_sessions):
count    690803.000000
mean          2.084525
std           0.898629
min           0.693147
25%           1.386294
50%           2.079442
75%           2.708050
max           8.404472
Name: log_total_sessions, dtype: float64

Saved phase5_buyers_features.csv and phase5_non_buyers_features.csv


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

output_path = ""

# ---------------------------
# Reload from Cell 1 outputs (safe to run independently)
# ---------------------------
buyers = pd.read_csv(output_path + "phase5_buyers_features.csv")
non_buyers = pd.read_csv(output_path + "phase5_non_buyers_features.csv")

print("Buyers loaded:", buyers.shape)
print("Non-buyers loaded:", non_buyers.shape)

# ---------------------------
# Feature sets for clustering
# Buyers: full RFM+ECP story
# Non-buyers: browsing-intensity story (no monetary signal exists)
# ---------------------------
buyer_cluster_cols = [
    'recency_days', 'log_total_sessions', 'purchase_rate',
    'log_estimated_avg_value', 'log_avg_session_duration', 'log_net_cart_activity'
]

non_buyer_cluster_cols = [
    'recency_days', 'log_total_sessions', 'log_total_events',
    'avg_unique_products', 'avg_unique_categories',
    'log_avg_session_duration', 'log_net_cart_activity'
]

print("\nBuyer clustering columns:", buyer_cluster_cols)
print("Non-buyer clustering columns:", non_buyer_cluster_cols)

# Confirm no missing columns before proceeding
missing_buyer_cols = [c for c in buyer_cluster_cols if c not in buyers.columns]
missing_nonbuyer_cols = [c for c in non_buyer_cluster_cols if c not in non_buyers.columns]
print("\nMissing buyer cols (should be empty):", missing_buyer_cols)
print("Missing non-buyer cols (should be empty):", missing_nonbuyer_cols)

# ---------------------------
# Sample for diagnostics only (5.3M rows too large for silhouette_score)
# Final fit in Cell 3 will use FULL data
# ---------------------------
SAMPLE_SIZE = 50000
buyers_sample = buyers.sample(n=min(SAMPLE_SIZE, len(buyers)), random_state=42)
non_buyers_sample = non_buyers.sample(n=min(SAMPLE_SIZE, len(non_buyers)), random_state=42)

def run_diagnostics_text(data, cols, group_name, k_range=range(2, 9)):
    X = data[cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print(f"\n{'='*60}")
    print(f"DIAGNOSTICS: {group_name}")
    print(f"{'='*60}")
    print(f"{'k':>3} | {'inertia':>14} | {'silhouette':>10}")
    print("-" * 35)

    results = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels, sample_size=10000, random_state=42)
        print(f"{k:>3} | {km.inertia_:>14.1f} | {sil:>10.4f}")
        results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil})

    results_df = pd.DataFrame(results)

    # Text-based "elbow" hint: % drop in inertia from previous k
    results_df['inertia_pct_drop'] = results_df['inertia'].pct_change() * -100
    print(f"\n{group_name}: % inertia drop at each step (look for where this flattens):")
    print(results_df[['k', 'inertia', 'inertia_pct_drop']].to_string(index=False))

    best_k_by_silhouette = results_df.loc[results_df['silhouette'].idxmax(), 'k']
    print(f"\n{group_name}: k with HIGHEST silhouette score = {int(best_k_by_silhouette)}")

    return results_df

print("\n" + "#"*60)
print("# BUYERS")
print("#"*60)
buyer_results = run_diagnostics_text(buyers_sample, buyer_cluster_cols, "Buyers")

print("\n" + "#"*60)
print("# NON-BUYERS")
print("#"*60)
nonbuyer_results = run_diagnostics_text(non_buyers_sample, non_buyer_cluster_cols, "NonBuyers")

# Save diagnostic tables as text/csv for reference
buyer_results.to_csv(output_path + "buyer_kmeans_diagnostics.csv", index=False)
nonbuyer_results.to_csv(output_path + "nonbuyer_kmeans_diagnostics.csv", index=False)
print(f"\nSaved diagnostics to {output_path}buyer_kmeans_diagnostics.csv and nonbuyer_kmeans_diagnostics.csv")

Buyers loaded: (690803, 24)
Non-buyers loaded: (4625758, 22)

Buyer clustering columns: ['recency_days', 'log_total_sessions', 'purchase_rate', 'log_estimated_avg_value', 'log_avg_session_duration', 'log_net_cart_activity']
Non-buyer clustering columns: ['recency_days', 'log_total_sessions', 'log_total_events', 'avg_unique_products', 'avg_unique_categories', 'log_avg_session_duration', 'log_net_cart_activity']

Missing buyer cols (should be empty): []
Missing non-buyer cols (should be empty): []

############################################################
# BUYERS
############################################################

DIAGNOSTICS: Buyers
  k |        inertia | silhouette
-----------------------------------
  2 |       211096.3 |     0.2728
  3 |       182040.4 |     0.1925
  4 |       162635.3 |     0.1839
  5 |       147032.3 |     0.1851
  6 |       134697.8 |     0.1873
  7 |       125772.3 |     0.1871
  8 |       118363.4 |     0.1786

Buyers: % inertia drop at each step (

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
import joblib

output_path = ""

# ---------------------------
# Reload full data (Cell 3 fits on FULL dataset, not the diagnostic sample)
# ---------------------------
buyers = pd.read_csv(output_path + "phase5_buyers_features.csv")
non_buyers = pd.read_csv(output_path + "phase5_non_buyers_features.csv")

print("Buyers loaded:", buyers.shape)
print("Non-buyers loaded:", non_buyers.shape)

buyer_cluster_cols = [
    'recency_days', 'log_total_sessions', 'purchase_rate',
    'log_estimated_avg_value', 'log_avg_session_duration', 'log_net_cart_activity'
]

non_buyer_cluster_cols = [
    'recency_days', 'log_total_sessions', 'log_total_events',
    'avg_unique_products', 'avg_unique_categories',
    'log_avg_session_duration', 'log_net_cart_activity'
]

# ---------------------------
# Final k chosen after diagnostics
# ---------------------------
K_BUYERS = 4
K_NON_BUYERS = 3

def fit_final_clusters(data, cols, k, group_name):
    X = data[cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)

    data = data.copy()
    data['cluster'] = labels

    print(f"\n{'='*70}")
    print(f"{group_name}: FINAL FIT (k={k})")
    print(f"{'='*70}")

    print(f"\n--- Cluster sizes ---")
    sizes = data['cluster'].value_counts().sort_index()
    for cl, cnt in sizes.items():
        print(f"Cluster {cl}: {cnt:,} users ({cnt/len(data)*100:.2f}%)")

    print(f"\n--- Cluster centroids (RAW, unscaled, mean per cluster) ---")
    centroid_table = data.groupby('cluster')[cols].mean()
    print(centroid_table.to_string())

    print(f"\n--- Cluster centroids (additional context columns, raw mean) ---")
    extra_cols = [c for c in ['total_sessions', 'purchase_sessions', 'purchase_rate',
                                'estimated_total_value', 'estimated_avg_value',
                                'recency_days', 'avg_session_duration', 'total_carts',
                                'avg_unique_products', 'avg_unique_categories',
                                'avg_brands', 'avg_hour_of_day']
                  if c in data.columns and c not in cols]
    if extra_cols:
        extra_table = data.groupby('cluster')[extra_cols].mean()
        print(extra_table.to_string())

    return data, km, scaler

buyers_clustered, buyer_kmeans, buyer_scaler = fit_final_clusters(
    buyers, buyer_cluster_cols, K_BUYERS, "BUYERS"
)
non_buyers_clustered, nonbuyer_kmeans, nonbuyer_scaler = fit_final_clusters(
    non_buyers, non_buyer_cluster_cols, K_NON_BUYERS, "NON-BUYERS"
)

# ---------------------------
# Tag group + offset non-buyer cluster IDs to avoid collisions when combined
# Buyer clusters: 0-3 | Non-buyer clusters: 100-102
# ---------------------------
buyers_clustered['segment_group'] = 'buyer'
non_buyers_clustered['segment_group'] = 'non_buyer'
non_buyers_clustered['cluster'] = non_buyers_clustered['cluster'] + 100

final_segmented = pd.concat([buyers_clustered, non_buyers_clustered], ignore_index=True)

print(f"\n{'='*70}")
print(f"FINAL COMBINED SEGMENTATION")
print(f"{'='*70}")
print("Final combined shape:", final_segmented.shape)
print("\nAll segment counts:")
print(final_segmented.groupby(['segment_group', 'cluster']).size().to_string())

# ---------------------------
# Save everything
# ---------------------------
final_segmented.to_csv(output_path + "phase5_user_segments.csv", index=False)
joblib.dump(buyer_kmeans, output_path + "buyer_kmeans.pkl")
joblib.dump(buyer_scaler, output_path + "buyer_scaler.pkl")
joblib.dump(nonbuyer_kmeans, output_path + "nonbuyer_kmeans.pkl")
joblib.dump(nonbuyer_scaler, output_path + "nonbuyer_scaler.pkl")

print(f"\nSaved phase5_user_segments.csv")
print(f"Saved buyer_kmeans.pkl, buyer_scaler.pkl, nonbuyer_kmeans.pkl, nonbuyer_scaler.pkl")
print(f"All files saved to: {output_path}")

Buyers loaded: (690803, 24)
Non-buyers loaded: (4625758, 22)

BUYERS: FINAL FIT (k=4)

--- Cluster sizes ---
Cluster 0: 189,832 users (27.48%)
Cluster 1: 121,662 users (17.61%)
Cluster 2: 219,992 users (31.85%)
Cluster 3: 159,317 users (23.06%)

--- Cluster centroids (RAW, unscaled, mean per cluster) ---
         recency_days  log_total_sessions  purchase_rate  log_estimated_avg_value  log_avg_session_duration  log_net_cart_activity
cluster                                                                                                                           
0            7.913898            3.014559       0.175602                 4.024697                  5.672526               2.103781
1           43.224014            1.190154       0.670951                 5.461028                  5.682018               0.569289
2           13.747186            2.260139       0.176679                 3.429709                  5.250419               0.706487
3           11.966733            1.4168

In [5]:
import pandas as pd

output_path = ""

# ---------------------------
# Load the combined segmented data from Cell 3
# ---------------------------
final_segmented = pd.read_csv(output_path + "phase5_user_segments.csv")
print("Loaded shape:", final_segmented.shape)

# ---------------------------
# Cluster ID -> human-readable name + short description
# Based on actual centroid values reviewed in Cell 3 output
# ---------------------------
cluster_name_map = {
    0:   "Frequent Browsers, Low Spend",
    1:   "High-Value Lapsed",
    2:   "Occasional Low-Value",
    3:   "High-Value Active",
    100: "Drive-by Visitors",
    101: "Engaged Browsers (No Cart)",
    102: "Cart Abandoners",
}

cluster_description_map = {
    0:   "Most active browsers among buyers; high session count but low purchase rate and low order value. Engaged but price-sensitive or undecided.",
    1:   "Same high spend and purchase-rate profile as 'High-Value Active', but hasn't been seen in ~43 days. Strongest win-back target.",
    2:   "Moderate browsing, lowest purchase rate and lowest spend of all buyer segments. Weakest buyer group.",
    3:   "Few sessions but high purchase rate and highest order value, and recently active. Best-performing current segment.",
    100: "Single short visit, minimal engagement. Lowest-intent browsing group.",
    101: "Long, exploratory sessions across many products/categories but never add to cart. Genuine researchers, not yet ready to buy.",
    102: "Highest activity among non-buyers and the only group that adds to cart; never completes checkout. Closest to conversion of any non-buyer segment.",
}

final_segmented['segment_name'] = final_segmented['cluster'].map(cluster_name_map)
final_segmented['segment_description'] = final_segmented['cluster'].map(cluster_description_map)

print("\nSegment name assignment check (should have NO unmapped/NaN names):")
print(final_segmented['segment_name'].isnull().sum(), "unmapped rows")
print("\nSegment distribution:")
print(final_segmented['segment_name'].value_counts().to_string())

# ---------------------------
# Build a clean per-segment profile summary table
# (one row per segment: name, group, size, %, and key stats)
# ---------------------------
profile_cols_buyer = ['recency_days', 'total_sessions', 'purchase_rate',
                       'estimated_avg_value', 'estimated_total_value',
                       'avg_session_duration', 'total_carts']
profile_cols_nonbuyer = ['recency_days', 'total_sessions', 'log_total_events',
                          'avg_unique_products', 'avg_session_duration', 'total_carts']

profile_rows = []
for cluster_id, name in cluster_name_map.items():
    subset = final_segmented[final_segmented['cluster'] == cluster_id]
    row = {
        'cluster_id': cluster_id,
        'segment_name': name,
        'segment_group': subset['segment_group'].iloc[0],
        'user_count': len(subset),
        'pct_of_total_users': len(subset) / len(final_segmented) * 100,
        'avg_recency_days': subset['recency_days'].mean(),
        'avg_total_sessions': subset['total_sessions'].mean(),
        'avg_purchase_rate': subset['purchase_rate'].mean() if subset['segment_group'].iloc[0] == 'buyer' else None,
        'avg_order_value': subset['estimated_avg_value'].mean() if subset['segment_group'].iloc[0] == 'buyer' else None,
        'avg_total_value': subset['estimated_total_value'].mean() if subset['segment_group'].iloc[0] == 'buyer' else None,
        'avg_session_duration_sec': subset['avg_session_duration'].mean(),
        'avg_carts': subset['total_carts'].mean(),
    }
    profile_rows.append(row)

profile_table = pd.DataFrame(profile_rows)
print(f"\n{'='*70}")
print("SEGMENT PROFILE SUMMARY TABLE")
print(f"{'='*70}")
print(profile_table.to_string(index=False))

# ---------------------------
# Business recommendation per segment (for dashboard display)
# ---------------------------
recommendation_map = {
    0:   "Promote bundle deals or limited-time offers to convert browsing into purchases.",
    1:   "Trigger win-back email/push campaign — discount code or 'we miss you' offer given high historical spend.",
    2:   "Low priority for active marketing spend; consider lightweight retargeting only.",
    3:   "Protect and reward — loyalty perks, early access, premium membership upsell.",
    100: "Low priority; minimal signal to act on from a single short visit.",
    101: "Retarget with product education content, reviews, or comparison guides to build buying confidence.",
    102: "Highest-priority retargeting — cart-abandonment email/discount within 24-48 hours.",
}
final_segmented['recommended_action'] = final_segmented['cluster'].map(recommendation_map)
profile_table['recommended_action'] = profile_table['cluster_id'].map(recommendation_map)

# ---------------------------
# Save outputs
# ---------------------------
final_segmented.to_csv(output_path + "phase5_user_segments_labeled.csv", index=False)
profile_table.to_csv(output_path + "phase5_segment_profile_summary.csv", index=False)

print(f"\nSaved phase5_user_segments_labeled.csv ({len(final_segmented):,} rows)")
print(f"Saved phase5_segment_profile_summary.csv ({len(profile_table)} rows — one per segment)")
print(f"All files saved to: {output_path}")

Loaded shape: (5316561, 26)

Segment name assignment check (should have NO unmapped/NaN names):
0 unmapped rows

Segment distribution:
segment_name
Drive-by Visitors               2523061
Engaged Browsers (No Cart)      1727653
Cart Abandoners                  375044
Occasional Low-Value             219992
Frequent Browsers, Low Spend     189832
High-Value Active                159317
High-Value Lapsed                121662

SEGMENT PROFILE SUMMARY TABLE
 cluster_id                 segment_name segment_group  user_count  pct_of_total_users  avg_recency_days  avg_total_sessions  avg_purchase_rate  avg_order_value  avg_total_value  avg_session_duration_sec  avg_carts
          0 Frequent Browsers, Low Spend         buyer      189832            3.570579          7.913898           24.230636           0.175602       128.869663      2865.323389                389.906404   9.487779
          1            High-Value Lapsed         buyer      121662            2.288359         43.224014       